# Лабораторная работа №1: Разведочный анализ данных (EDA)
## Исследование соревновательных матчей CS2 (Tier 1–3)

**Цель работы:**
1. Провести предобработку и очистку данных киберспортивной телеметрии Counter-Strike 2.
2. Сформировать новые агрегированные признаки (Feature Engineering) командного уровня.
3. Выполнить разведочный анализ данных (EDA) с помощью визуализаций распределений, ящиков с усами и корреляционных матриц.
4. Статистически проверить гипотезу об однородности длительности матчей по соревновательным картам.

### 1. Импорт библиотек и первичная настройка

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)

### 2. Загрузка данных и Feature Engineering
Датасет содержит телеметрию на уровне игроков. Агрегируем индивидуальный урон (`adr`) и фраги (`kills`) в командные метрики, а также рассчитаем общую продолжительность игры в раундах (`total_rounds`) и разницу по среднему урону (`adr_diff`).

In [ ]:
df = pd.read_csv("cs2_all_tiers_games.csv")

# Формируем списки колонок для игроков каждой команды
t1_adr_cols = [f"team1_player{i}_adr" for i in range(1, 6)]
t2_adr_cols = [f"team2_player{i}_adr" for i in range(1, 6)]
t1_kills_cols = [f"team1_player{i}_kills" for i in range(1, 6)]
t2_kills_cols = [f"team2_player{i}_kills" for i in range(1, 6)]

# Агрегированные метрики команд
df["team1_mean_adr"] = df[t1_adr_cols].mean(axis=1)
df["team2_mean_adr"] = df[t2_adr_cols].mean(axis=1)
df["team1_total_kills"] = df[t1_kills_cols].sum(axis=1)
df["team2_total_kills"] = df[t2_kills_cols].sum(axis=1)

df["total_rounds"] = df["score1_game"] + df["score2_game"]
df["adr_diff"] = df["team1_mean_adr"] - df["team2_mean_adr"]

### 3. Очистка данных от аномалий (Data Cleaning)
Отбираем топ-7 соревновательных карт активного пула и фильтруем технические нули (`total_rounds >= 13`), вызванные техническими поражениями или несыгранными картами.

In [ ]:
top_maps = df["map_name"].value_counts().head(7).index

# Фильтрация аномалий и отбор активного маппула
df_filtered = df[(df["map_name"].isin(top_maps)) & (df["total_rounds"] >= 13)].copy()

print("Статистика после очистки:")
display(df_filtered[["total_rounds", "team1_mean_adr", "adr_diff"]].describe().T[["mean", "std", "50%", "min", "max"]])

### 4. Визуализация данных (EDA)

#### График 1: Распределение сыгранных матчей по картам
Оценка популярности карт в профессиональном соревновательном пуле.

In [ ]:
plt.figure(figsize=(9, 4))
sns.countplot(
    data=df_filtered,
    x="map_name",
    order=top_maps,
    hue="map_name",
    palette="viridis",
    legend=False,
)
plt.title("Распределение сыгранных матчей по картам")
plt.xlabel("Карта")
plt.ylabel("Количество игр")
plt.tight_layout()
plt.show()

#### График 2: Длительность игр (раундов на карте)
Гистограмма и плотность распределения общего числа раундов. Красной пунктирной линией отмечен лимит основного времени (24 раунда в MR12).

In [ ]:
plt.figure(figsize=(9, 4))
sns.histplot(df_filtered["total_rounds"], kde=True, discrete=True, color="#3498db")
plt.axvline(24, color="red", linestyle="--", label="Максимум основного времени (MR12: 24)")
plt.title("Распределение длительности игр (раундов на карте)")
plt.xlabel("Суммарное число раундов")
plt.ylabel("Частота")
plt.legend()
plt.tight_layout()
plt.show()

#### График 3: Разброс раундов на соревновательных картах
Диаграмма размаха (Boxplot) для проверки стабильности длительности на разных картах и выявления овертаймов.

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(
    data=df_filtered,
    x="map_name",
    y="total_rounds",
    hue="map_name",
    palette="Set2",
    legend=False,
)
plt.title("Разброс раундов на соревновательных картах")
plt.xlabel("Карта")
plt.ylabel("Всего раундов")
plt.tight_layout()
plt.show()

#### График 4: Разница среднего командного ADR и исход карты
Оценка разделения победителей и проигравших по преимуществу в среднем огневом уроне (`adr_diff`).

In [ ]:
plt.figure(figsize=(7, 5))
sns.boxplot(
    data=df_filtered,
    x="team1_win",
    y="adr_diff",
    hue="team1_win",
    palette=["#e74c3c", "#2ecc71"],
    legend=False,
)
plt.xticks([0, 1], ["Поражение Team 1", "Победа Team 1"])
plt.title("Разница среднего командного ADR и исход карты")
plt.xlabel("Результат матча")
plt.ylabel("Разница по ADR")
plt.tight_layout()
plt.show()

#### График 5: Матрица корреляций признаков
Исследование линейных связей между числовыми показателями раундов, урона и фактом победы.

In [ ]:
corr_cols = [
    "score1_game", "score2_game", "total_rounds",
    "team1_win", "team1_mean_adr", "team2_mean_adr", "adr_diff"
]
plt.figure(figsize=(8, 6))
sns.heatmap(
    df_filtered[corr_cols].corr(),
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
)
plt.title("Матрица корреляций ключевых признаков")
plt.tight_layout()
plt.show()

### 5. Статистическая проверка гипотез
**Гипотеза $H_0$:** Продолжительность игры (`total_rounds`) статистически однородна по всем соревновательным картам.
**Альтернатива $H_1$:** Хотя бы на одной карте среднее/медианное число раундов значимо отличается.

Поскольку распределение раундов дискретно и имеет овертайм-выбросы, основным критерием выбран непараметрический тест **Краскела — Уоллиса (Kruskal-Wallis)**.

In [ ]:
map_samples = [
    group["total_rounds"].dropna().values
    for _, group in df_filtered.groupby("map_name")
]

# 1. Параметрический критерий (One-way ANOVA)
f_stat, p_val_f = stats.f_oneway(*map_samples)

# 2. Непараметрический критерий (Kruskal-Wallis)
kw_stat, p_val_kw = stats.kruskal(*map_samples)

print(f"One-way ANOVA:  F = {f_stat:.4f}, p-value = {p_val_f:.4f}")
print(f"Kruskal-Wallis: H = {kw_stat:.4f}, p-value = {p_val_kw:.4f}")

### 6. Итоговые выводы
1. **Маппул:** Карты **Ancient** и **Mirage** лидируют по частоте выбора (~2200 игр), тогда как **Overpass** наименее востребован ввиду ротации пула Valve.
2. **Длительность матчей:** Медианное количество раундов составляет **21 раунд**. Тест Краскела — Уоллиса ($H = 7.03$, $p = 0.318 > 0.05$) подтверждает нулевую гипотезу: соревновательные карты статистически сбалансированы по таймингу и плотности борьбы.
3. **Решающий фактор:** Разница среднего урона (`adr_diff`) демонстрирует высокую положительную корреляцию с победой команды ($r = 0.68$), что делает её ключевым кандидатом в предикторы для последующих моделей классификации.